# Tasks 3, 4, and 5: Future Forecasting, Portfolio Optimization, and Backtesting

In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_DIR, FIGURES_DIR, REPORTS_DIR, TEST_START_DATE
from src.data_loader import fetch_all_assets, pivot_adjusted_close
from src.preprocessing import clean_all_assets, calculate_daily_returns
from src.modeling import fit_arima_forecast
from src.forecasting import choose_future_horizon, forecast_future_arima, infer_forecast_return
from src.portfolio import annualized_expected_returns, annualized_covariance_matrix, efficient_frontier, optimize_max_sharpe, optimize_min_volatility, recommendation_table
from src.backtesting import isolate_backtest_window, backtest_strategy_vs_benchmark, cumulative_returns
from src.visualization import plot_future_forecast, plot_covariance_heatmap, plot_efficient_frontier, plot_backtest_cumulative_returns

In [ ]:
prices_path = DATA_DIR / "adjusted_close_prices.csv"
if prices_path.exists():
    prices = pd.read_csv(prices_path, index_col="Date", parse_dates=True)
else:
    prices = pivot_adjusted_close(clean_all_assets(fetch_all_assets()))
    prices.to_csv(prices_path)
returns = calculate_daily_returns(prices)
tsla = prices["TSLA"].dropna()

## Task 3: 6-Month Future Forecast with Confidence Intervals

In [ ]:
arima_results, train, test, test_forecast, metrics = fit_arima_forecast(tsla, split_date=TEST_START_DATE, order=(1,1,1))
horizon_steps = choose_future_horizon(months=6)
future_forecast = forecast_future_arima(arima_results, steps=horizon_steps)
future_forecast.to_csv(REPORTS_DIR / "task3_future_forecast_ci.csv")
plot_future_forecast(train, test, test_forecast, future_forecast, FIGURES_DIR / "task3_future_forecast_with_ci.png");
future_forecast.head()

## Task 4: Portfolio Optimization
TSLA expected return is taken from the forecast-implied annualized return, while BND and SPY use historical annualized returns. The covariance matrix comes from daily returns.

In [ ]:
tsla_expected_return = infer_forecast_return(float(tsla.iloc[-1]), float(future_forecast["forecast"].iloc[-1]), horizon_steps)
expected_returns = annualized_expected_returns(returns, tsla_forecast_return=tsla_expected_return)
covariance = annualized_covariance_matrix(returns)
frontier = efficient_frontier(expected_returns, covariance, points=60)
max_sharpe = optimize_max_sharpe(expected_returns, covariance)
min_vol = optimize_min_volatility(expected_returns, covariance)
plot_covariance_heatmap(covariance, FIGURES_DIR / "task4_covariance_heatmap.png");
plot_efficient_frontier(frontier, max_sharpe, min_vol, FIGURES_DIR / "task4_efficient_frontier.png");
recommendation_table(max_sharpe)

## Task 5: Backtesting Against 60% SPY / 40% BND Benchmark
The strategy uses the Task 4 maximum-Sharpe weights as a static hold portfolio over the final-year held-out window.

In [ ]:
backtest_returns = isolate_backtest_window(returns)
comparison_returns, backtest_metrics = backtest_strategy_vs_benchmark(backtest_returns, max_sharpe["weights"])
cumulative = cumulative_returns(comparison_returns)
plot_backtest_cumulative_returns(cumulative, FIGURES_DIR / "task5_strategy_vs_benchmark.png");
backtest_metrics